In [1]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image
from model import Ganerator_MNIST, VAE_Encoder_MNIST
# import wandb 
import torch, math
import torch.nn as nn


In [2]:
def timeembedding(t, dim):
    # t: (B,) int64
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(0, half, device=t.device).float() / (half - 1)
    )
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0,1))
    return emb  # (B, dim)

In [4]:
t = torch.tensor([300], dtype=torch.long)
print(timeembedding(t, 64))

tensor([[-0.9998,  0.1632,  0.7869, -0.4897, -0.2989, -0.9324,  0.1911, -0.2093,
          0.4095,  0.9630,  0.3274, -0.9102,  0.8065,  0.0221, -0.9996, -0.3324,
          0.5276,  0.9392,  0.9897,  0.8726,  0.7089,  0.5525,  0.4213,  0.3175,
          0.2378,  0.1774,  0.1321,  0.0983,  0.0731,  0.0543,  0.0404,  0.0300,
         -0.0221, -0.9866, -0.6171, -0.8719, -0.9543,  0.3615,  0.9816,  0.9779,
         -0.9123, -0.2696, -0.9449,  0.4142, -0.5913,  0.9998, -0.0278, -0.9431,
         -0.8495, -0.3433,  0.1429,  0.4884,  0.7053,  0.8335,  0.9069,  0.9482,
          0.9713,  0.9841,  0.9912,  0.9952,  0.9973,  0.9985,  0.9992,  0.9996]])


In [2]:
epochs=20
batch_size=128
z_dim=64
lr=2e-4
device="cuda"
save_every=5
header = 1

In [3]:
tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
ds = datasets.MNIST(root=r"D:\Python\SemCom\saved_data", train=True, download=True, transform=tfm)
dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

In [ ]:
G = Ganerator_MNIST(z_dim=z_dim).to(device)
G.load_state_dict(torch.load("./save_model/G_wgangp_C.pth", map_location=device))
G.eval()
for p in G.parameters():
    p.requires_grad = False

E = VAE_Encoder_MNIST(latent_dim=z_dim).to(device)
optE = torch.optim.Adam(E.parameters(), lr=lr)

fixed_x, _ = next(iter(dl))
fixed_x = fixed_x[:64].to(device)
fixed_x_cpu = fixed_x.detach().cpu()
save_image((fixed_x_cpu + 1) / 2, "./save_data/fixed_x.png",nrow=8)

step = 0